In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# -----------------------------
# INITIAL EXPLORATION
# -----------------------------

print(unemployment.head())
print(unemployment.info())
print(unemployment.describe())


# -----------------------------
# CONTINENT COUNTS
# -----------------------------

print(unemployment["continent"].value_counts())


# -----------------------------
# HISTOGRAM (GLOBAL 2021)
# -----------------------------

sns.histplot(data=unemployment, x="2021", binwidth=1)
plt.show()


# -----------------------------
# REMOVE OCEANIA
# -----------------------------

not_oceania = ~unemployment["continent"].isin(["Oceania"])
print(unemployment[not_oceania])


# -----------------------------
# BOXPLOT + MIN/MAX
# -----------------------------

print(unemployment["2021"].min(), unemployment["2021"].max())

sns.boxplot(data=unemployment, x="2021", y="continent")
plt.show()


# -----------------------------
# GROUPBY SUMMARY
# -----------------------------

print(unemployment[["2019", "2020"]].agg(["mean", "std"]))
print(unemployment.groupby("continent")[["2019", "2020"]].agg(["mean", "std"]))


# -----------------------------
# NAMED AGGREGATIONS
# -----------------------------

continent_summary = unemployment.groupby("continent").agg(
    mean_rate_2021=("2021", "mean"),
    std_rate_2021=("2021", "std")
)

print(continent_summary)


# -----------------------------
# BAR PLOT (MEAN BY CONTINENT)
# -----------------------------

sns.barplot(data=unemployment, x="continent", y="2021")
plt.show()


# -----------------------------
# MISSING DATA (PLANES)
# -----------------------------

print(planes.isna().sum())

threshold = len(planes) * 0.05
cols_to_drop = planes.columns[planes.isna().sum() <= threshold]

planes.drop(columns=cols_to_drop, inplace=True)

print(planes.isna().sum())


# -----------------------------
# ADDITIONAL_INFO + PRICE ANALYSIS
# -----------------------------

print(planes["Additional_Info"].value_counts())

sns.boxplot(data=planes, x="Airline", y="Price")
plt.xticks(rotation=90)
plt.show()


# -----------------------------
# IMPUTE PRICE BY AIRLINE
# -----------------------------

airline_prices = planes.groupby("Airline")["Price"].median()
prices_dict = airline_prices.to_dict()

planes["Price"] = planes["Price"].fillna(planes["Airline"].map(prices_dict))


# -----------------------------
# UNIQUE VALUES CHECK
# -----------------------------

non_numeric = planes.select_dtypes("object")

for col in non_numeric.columns:
    print(f"Number of unique values in {col} column:", non_numeric[col].nunique())


# -----------------------------
# FLIGHT DURATION CATEGORIES
# -----------------------------

flight_categories = ["Short-haul", "Medium", "Long-haul"]

short_flights = "^0h|^1h|^2h|^3h|^4h"
medium_flights = "^5h|^6h|^7h|^8h|^9h"
long_flights = "10h|11h|12h|13h|14h|15h|16h"

conditions = [
    planes["Duration"].str.contains(short_flights),
    planes["Duration"].str.contains(medium_flights),
    planes["Duration"].str.contains(long_flights)
]

planes["Duration_Category"] = np.select(
    conditions,
    flight_categories,
    default="Extreme duration"
)

sns.countplot(data=planes, x="Duration_Category")
plt.show()


# -----------------------------
# CLEAN DURATION COLUMN
# -----------------------------

print(planes["Duration"].head())

planes["Duration"] = planes["Duration"].str.replace("h", "")
planes["Duration"] = planes["Duration"].astype(float)

sns.histplot(data=planes, x="Duration")
plt.show()


# -----------------------------
# GROUPED STATS (TRANSFORM)
# -----------------------------

planes["airline_price_st_dev"] = planes.groupby("Airline")["Price"].transform("std")

planes["airline_median_duration"] = planes.groupby("Airline")["Duration"].transform("median")

planes["price_destination_mean"] = planes.groupby("Destination")["Price"].transform("mean")


# -----------------------------
# OUTLIERS (PRICE)
# -----------------------------

sns.histplot(data=planes, x="Price")
plt.show()

print(planes["Duration"].describe())


price_seventy_fifth = planes["Price"].quantile(0.75)
price_twenty_fifth = planes["Price"].quantile(0.25)

prices_iqr = price_seventy_fifth - price_twenty_fifth

upper = price_seventy_fifth + (1.5 * prices_iqr)
lower = price_twenty_fifth - (1.5 * prices_iqr)

planes = planes[(planes["Price"] > lower) & (planes["Price"] < upper)]

print(planes["Price"].describe())